# 1.2 - How Models Learn: Loss, Gradients, and Training Loops

**Module 1 - Math Prerequisites** | Netsetos GenAI Engineering

This notebook builds the mechanics of learning from the ground up: how a model measures its own error (cross-entropy loss), which direction reduces that error (the gradient), and how repeated small steps in that direction (gradient descent) fit parameters to data. You start with hand-written NumPy math, then graduate to PyTorch autograd and a full production-style training loop.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

Install the two numerical libraries this lesson depends on: NumPy for the from-scratch math and PyTorch for automatic differentiation.

In [ ]:
!pip install -q numpy torch

**Explanation**

A package-installation cell. It quietly (`-q`) pulls NumPy and PyTorch into the runtime so the later cells can import them. Nothing is computed here.

**How the code works - step by step**
- `!pip install -q numpy torch` shells out to pip and installs both packages in quiet mode.
- **In one line:** prepares the environment so every following cell can run.

**What you'll see:** (no output - environment prep)

## Imports

Bring the installed libraries into scope. NumPy powers the manual loss and gradient math; the three PyTorch modules power autograd, layers, and functional helpers.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

**Explanation**

A plain import cell that binds the names used throughout the notebook. `nn` gives layer/loss classes, `F` gives functional ops, and `np`/`torch` are the two array backends.

**How the code works - step by step**
- `import numpy as np` - array math for the from-scratch skills.
- `import torch` - the tensor library with autograd.
- `import torch.nn as nn` - layers, containers, and loss classes.
- `import torch.nn.functional as F` - stateless functional versions of those ops.
- **In one line:** loads every library the rest of the cells reference.

**What you'll see:** (no output - environment prep)

## 1 - Cross-entropy loss

A classifier outputs raw scores (logits); cross-entropy turns them into a single number measuring how surprised the model is by the correct answer. Here you implement softmax and cross-entropy by hand in NumPy to see exactly what that number is made of.

In [ ]:
def softmax(x):
    s = x - np.max(x)
    e = np.exp(s)
    return e / e.sum()

def cross_entropy(logits, true_idx):
    probs = softmax(logits)
    return -np.log(probs[true_idx] + 1e-12)

logits = np.array([2.0, 1.0, 0.1])
loss = cross_entropy(logits, true_idx=0)
print(f'probs = {softmax(logits).round(3)}')
print(f'loss  = {loss:.3f}')

**Explanation**

A from-scratch implementation of the softmax-then-cross-entropy pipeline. `softmax` converts logits to a probability distribution, and `cross_entropy` reads off the negative log-probability the model assigned to the true class - lower means the model was more confident and correct.

**How the code works - step by step**
- `softmax(x)` subtracts the max for numerical stability, exponentiates, and normalizes so the outputs sum to 1.
- `cross_entropy(logits, true_idx)` softmaxes the logits and returns `-log(prob of the true class)`, with a `1e-12` floor to avoid `log(0)`.
- `logits = [2.0, 1.0, 0.1]` with `true_idx=0` scores a case where the true class already has the highest logit.
- The two `print`s show the probability vector and the resulting loss.
- **In one line:** loss is the negative log-probability the model gave the right answer.

**What you'll see:** Prints the probability distribution `probs = [0.659 0.242 0.099]` and `loss = 0.417` - a modest loss because the correct class (index 0) already carries the most probability.

## 2 - Gradient (manual)

Before automating anything, compute a gradient by hand for a simple parabola. The gradient tells you both the direction and steepness of the loss surface at your current parameter value - the single most important quantity in training.

In [ ]:
def loss(w): return (w - 3) ** 2
def grad(w): return 2 * (w - 3)

w = 0.0
print(f'L({w}) = {loss(w)}, grad = {grad(w)}')

**Explanation**

A minimal analytic example: a quadratic loss `(w-3)^2` and its exact derivative `2(w-3)`. Evaluating both at `w=0` shows how the gradient's sign and magnitude point back toward the minimum at `w=3`.

**How the code works - step by step**
- `loss(w)` defines the parabola `(w - 3)**2`, minimized at `w = 3`.
- `grad(w)` returns its closed-form derivative `2 * (w - 3)`.
- `w = 0.0` picks a starting point away from the minimum.
- The `print` reports the loss and gradient at that point.
- **In one line:** the negative gradient sign tells you which way to move `w` to shrink the loss.

**What you'll see:** Prints `L(0.0) = 9.0, grad = -6.0` - the loss is 9 and the negative gradient signals that increasing `w` (toward 3) will reduce it.

## 3 - Gradient descent

Gradient descent is just the previous step repeated: nudge `w` a little against the gradient, over and over. The learning rate controls the step size - and this cell shows how the same loop converges, nearly converges, or blows up depending on that one number.

In [ ]:
def descend(w_init, lr, steps=30):
    w = w_init
    for _ in range(steps):
        w = w - lr * grad(w)
    return w

for lr in [0.01, 0.1, 1.5]:
    final_w = descend(0.0, lr)
    print(f'LR={lr:4.2f}  final w={final_w:.2e}')

**Explanation**

A hand-rolled gradient-descent loop run at three learning rates. It reuses the `grad` from the previous cell and iterates 30 updates, revealing that too-small a rate under-shoots the minimum while too-large a rate diverges.

**How the code works - step by step**
- `descend(w_init, lr, steps=30)` loops `steps` times, each time applying `w = w - lr * grad(w)`.
- The `for lr in [0.01, 0.1, 1.5]` sweep runs the same descent at a slow, a good, and a reckless learning rate.
- Each `print` reports the final `w` after 30 steps in scientific notation.
- **In one line:** learning rate is the make-or-break dial - too small crawls, too big explodes.

**What you'll see:** Prints one line per rate: `LR=0.01` lands short of 3 (around 1.4), `LR=0.10` converges to ~3.00, and `LR=1.50` diverges to a huge value - overshooting past the minimum on every step.

## 4 - Backpropagation via PyTorch autograd

Hand-deriving gradients does not scale to real networks. PyTorch's autograd records every operation and computes all gradients automatically when you call `.backward()`. This cell trains a small multi-layer network on a single example to watch the loss fall.

> **Needs GPU** (optional - runs fine on CPU too).

In [ ]:
torch.manual_seed(0)
model = nn.Sequential(
    nn.Linear(2, 4), nn.ReLU(),
    nn.Linear(4, 2), nn.ReLU(),
    nn.Linear(2, 1),
)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
loss_fn = nn.MSELoss()
x = torch.tensor([[0.5, 0.3]])
y = torch.tensor([[1.0]])

for step in range(200):
    optimizer.zero_grad()
    y_hat = model(x)
    l = loss_fn(y_hat, y)
    l.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

print(f'final loss = {l.item():.4f}')
print(f'final y_hat = {y_hat.item():.3f}')

**Explanation**

A real (if tiny) PyTorch training loop. It builds a 3-layer MLP, then repeatedly forward-passes one input, computes MSE loss, backpropagates gradients with `.backward()`, clips them, and updates weights with AdamW - the same skeleton used to train large models.

**How the code works - step by step**
- `torch.manual_seed(0)` fixes initialization so results are reproducible.
- `nn.Sequential(...)` stacks Linear->ReLU->Linear->ReLU->Linear into a small network.
- `AdamW` is the optimizer; `nn.MSELoss()` measures squared error against the target `y`.
- The `for step in range(200)` loop zeroes gradients, forward-passes, computes loss, calls `l.backward()` to autograd all gradients, clips them to `max_norm=1.0`, then `optimizer.step()` updates weights.
- The final `print`s report the converged loss and prediction.
- **In one line:** autograd computes every gradient for you, so the loop just calls `.backward()` and steps.

**What you'll see:** After 200 steps it prints a near-zero `final loss` and a `final y_hat` close to the target of 1.0 - the network has fit the single training point.

## 5 - Full production training loop

This is what a realistic training step looks like once every best practice is wired in: a proper optimizer, a learning-rate schedule, gradient clipping, and label smoothing on a classification loss. It trains on fresh random batches to demonstrate the moving parts rather than to solve a real task.

In [ ]:
model = nn.Sequential(nn.Linear(10, 20), nn.ReLU(), nn.Linear(20, 2))
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)
loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

for step in range(100):
    x = torch.randn(16, 10)
    y = torch.randint(0, 2, (16,))
    optimizer.zero_grad(set_to_none=True)
    logits = model(x)
    loss = loss_fn(logits, y)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    scheduler.step()
print(f'final loss = {loss.item():.4f}')

**Explanation**

A production-flavored classification training loop. On top of the basic pattern it adds `CosineAnnealingLR` scheduling, `label_smoothing`, `set_to_none=True` zeroing, and gradient clipping - the ingredients you would actually ship. Batches are random, so this showcases the mechanics rather than learning a signal.

**How the code works - step by step**
- The `nn.Sequential` MLP maps 10 features to 2 class logits.
- `AdamW` plus `CosineAnnealingLR(T_max=100)` decays the learning rate smoothly over training.
- `CrossEntropyLoss(label_smoothing=0.1)` softens the targets to curb overconfidence.
- Inside `for step in range(100)`: draw a random batch `x`/`y`, zero grads with `set_to_none=True`, forward, compute loss, `backward()`, clip to `max_norm=1.0`, `optimizer.step()`, then `scheduler.step()`.
- The final `print` reports the last loss value.
- **In one line:** the same core loop, now dressed with the scheduling, clipping, and smoothing real training runs use.

**What you'll see:** Prints a single `final loss` line hovering near the label-smoothed floor (roughly 0.6-0.7) - since the batches are pure noise, the loss stays around chance rather than dropping to zero.

You've now traced the full learning stack: cross-entropy turns predictions into a single error number, the gradient points toward lower error, and gradient descent takes repeated steps down that slope - too small a learning rate crawls, too large a rate diverges. PyTorch autograd computes those gradients automatically for arbitrary networks, and the final loop layers in the production essentials (AdamW, weight decay, LR scheduling, gradient clipping, label smoothing) that every real training run relies on. Everything a large model does when it learns is this same loop, scaled up.